# Removing top prediction on the 10 trim


In [1]:
import pandas as pd

In [2]:
def load_tsv(tsv_path):
    """Load a merged ensemble TSV (has a header, arbitrary number of conf_* columns)."""
    compression = 'gzip' if tsv_path.endswith('.gz') else None
    # load with pandas, specifying the header names protein_id, go_term, confidnece
    df = pd.read_csv(tsv_path, compression=compression, sep='\t', header=None)
    df.columns = ['protein_id', 'go_term'] + [f'conf_{i}' for i in range(1, df.shape[1]-1)]
    return df

In [3]:
sequence_10_df = load_tsv('/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/clean/CAFA_trim_thr-10/test_submission.tsv')
print("SEQUENCE TEAM TEST TRIM THR 10:")
display(sequence_10_df.head())

sequence_50_df = load_tsv('/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/clean/CAFA_trim_thr-50/test_submission.tsv')
print("SEQUENCE TEAM TEST TRIM THR 50:")
display(sequence_50_df.head())

SEQUENCE TEAM TEST TRIM THR 10:


,protein_id,go_term,conf_1
0,Q4QQW4,GO:0032501,0.111
1,Q4QQW4,GO:0050794,0.311
2,Q4QQW4,GO:0009987,0.648
3,Q4QQW4,GO:0051716,0.148
4,Q4QQW4,GO:0065007,0.428


SEQUENCE TEAM TEST TRIM THR 50:


,protein_id,go_term,conf_1
0,Q4QQW4,GO:0009987,0.648
1,Q4QQW4,GO:0050896,0.582
2,P35994,GO:0009987,0.544
3,P62968,GO:0050896,0.511
4,P49897,GO:0009987,0.575


In [4]:
# Remove top row of sequence_10_df
sequence_10_df = sequence_10_df.iloc[1:].reset_index(drop=True)
display(sequence_10_df.head())

,protein_id,go_term,conf_1
0,Q4QQW4,GO:0050794,0.311
1,Q4QQW4,GO:0009987,0.648
2,Q4QQW4,GO:0051716,0.148
3,Q4QQW4,GO:0065007,0.428
4,Q4QQW4,GO:0007154,0.103


In [5]:
# save to output_path with no header
output_path_10 = '/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/clean/CAFA_trim_thr-10/test_submission_no_first_element.tsv'
sequence_10_df.to_csv(output_path_10, sep='\t', index=False, header=False)

test = load_tsv(output_path_10)
display(test.head())

,protein_id,go_term,conf_1
0,Q4QQW4,GO:0050794,0.311
1,Q4QQW4,GO:0009987,0.648
2,Q4QQW4,GO:0051716,0.148
3,Q4QQW4,GO:0065007,0.428
4,Q4QQW4,GO:0007154,0.103


In [6]:
display(load_tsv("/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/sequence_team/clean/CAFA_trim_thr-10/best_sequence_test_trim10.tsv"))

,protein_id,go_term,conf_1
0,Q4QQW4,GO:0050794,0.311
1,Q4QQW4,GO:0009987,0.648
2,Q4QQW4,GO:0051716,0.148
3,Q4QQW4,GO:0065007,0.428
4,Q4QQW4,GO:0007154,0.103
...,...,...,...
18318310,A2ASS6,GO:0005200,0.553
18318311,A2ASS6,GO:0098918,0.113
18318312,A2ASS6,GO:0008307,0.399
18318313,A2ASS6,GO:0031433,0.102


In [3]:
import pandas as pd

def analyze_predictions_per_method(tsv_path):
    """Loads the TSV and returns summary stats for predictions per protein, broken down by method."""
    compression = 'gzip' if tsv_path.endswith('.gz') else None
    
    # Read the file
    df = pd.read_csv(tsv_path, compression=compression, sep='\t')
    
    # If the first column isn't named 'protein_id', the file likely lacks a header.
    # We will assign the columns based on your provided structure.
    if 'protein_id' not in df.columns:
        df = pd.read_csv(tsv_path, compression=compression, sep='\t', header=None)
        # Assuming 6 columns: protein_id, go_term, the 3 confs, and the label
        df.columns = ['protein_id', 'go_term', 'conf_sequence', 'conf_structure', 'conf_protgoat', 'label']

    # Find all columns that represent model confidences
    conf_cols = [col for col in df.columns if col.startswith('conf_')]
    
    print(f"\n{'='*50}")
    print(f"FILE: {tsv_path.split('/')[-1]}")
    print(f"{'='*50}")
    
    for col in conf_cols:
        # A prediction counts ONLY if it is greater than 0 (and not NaN)
        has_prediction = (df[col].notna()) & (df[col] > 0.0)
        
        # Group by protein_id and sum the True/False values (True counts as 1)
        predictions_per_protein = has_prediction.groupby(df['protein_id']).sum()
        
        print(f"\n--- Method: {col.upper()} ---")
        print(f"  Total Predictions Overall: {has_prediction.sum():,}")
        print(f"  Min per protein:    {predictions_per_protein.min()}")
        print(f"  Max per protein:    {predictions_per_protein.max()}")
        print(f"  Mean per protein:   {round(predictions_per_protein.mean(), 2)}")
        print(f"  Median per protein: {predictions_per_protein.median()}")

# 1. Define your file paths
test_path = '/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/final_data/test_merged.tsv'
train_path = '/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/final_data/train_merged.tsv'

# 2. Run the analysis
print("Loading and analyzing data...")
analyze_predictions_per_method(test_path)
analyze_predictions_per_method(train_path)

Loading and analyzing data...

FILE: test_merged.tsv

--- Method: CONF_SEQUENCE ---
  Total Predictions Overall: 3,968,907
  Min per protein:    0
  Max per protein:    232
  Mean per protein:   27.96
  Median per protein: 23.0

--- Method: CONF_STRUCTURE ---
  Total Predictions Overall: 122,634,206
  Min per protein:    0
  Max per protein:    2151
  Mean per protein:   864.02
  Median per protein: 884.0

--- Method: CONF_PROTGOAT ---
  Total Predictions Overall: 9,448,778
  Min per protein:    0
  Max per protein:    879
  Mean per protein:   66.57
  Median per protein: 52.0

FILE: train_merged.tsv

--- Method: CONF_SEQUENCE ---
  Total Predictions Overall: 3,625,962
  Min per protein:    0
  Max per protein:    232
  Mean per protein:   25.49
  Median per protein: 21.0

--- Method: CONF_STRUCTURE ---
  Total Predictions Overall: 121,173,554
  Min per protein:    0
  Max per protein:    2123
  Mean per protein:   851.86
  Median per protein: 931.0

--- Method: CONF_PROTGOAT ---
  Tot

In [1]:
import pandas as pd
import os

def trim_structure_and_analyze(input_path, output_merged, output_struct, threshold=0.05):
    print(f"{'='*50}")
    print(f"Processing: {os.path.basename(input_path)}")
    print(f"{'='*50}")
    
    # 1. Load Data
    compression = 'gzip' if input_path.endswith('.gz') else None
    # Added low_memory=False to prevent mixed-type warnings on large files
    df = pd.read_csv(input_path, compression=compression, sep='\t', header=None, low_memory=False)
    
    # 2. Dynamically check the number of columns
    num_cols = df.shape[1]
    has_labels = (num_cols == 6)
    
    if has_labels:
        df.columns = ['protein_id', 'go_term', 'conf_sequence', 'conf_structure', 'conf_protgoat', 'label']
        numeric_cols = ['conf_sequence', 'conf_structure', 'conf_protgoat', 'label']
    elif num_cols == 5:
        df.columns = ['protein_id', 'go_term', 'conf_sequence', 'conf_structure', 'conf_protgoat']
        numeric_cols = ['conf_sequence', 'conf_structure', 'conf_protgoat']
    else:
        raise ValueError(f"Unexpected number of columns: {num_cols}. Expected 5 or 6.")
    
    # --- NEW STEP: Force columns to numeric ---
    # This fixes the TypeError. errors='coerce' turns strings (like headers or "NA") into NaN
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
        
    # 3. Analyze Positive Labels (ONLY if the label column exists)
    if has_labels:
        positive_labels = df[df['label'] == 1.0]
        labels_per_protein = positive_labels.groupby('protein_id').size()
        
        print("--- Label Statistics ---")
        print(f"  Total distinct proteins with labels: {len(labels_per_protein):,}")
        print(f"  Average positive labels per protein: {round(labels_per_protein.mean(), 2)}")
        print(f"  Median positive labels per protein:  {labels_per_protein.median()}")
        print(f"  Max positive labels on one protein:  {labels_per_protein.max()}")
    else:
        print("--- Label Statistics ---")
        print("  No 'label' column found in this file (likely test data). Skipping label stats.")
    
    # 4. Calculate "Before" stats for Structure
    total_struct_before = (df['conf_structure'] > 0).sum()
    rows_before = len(df)
    
    # 5. Apply the Threshold
    # If structure confidence is less than the threshold, zero it out
    df.loc[df['conf_structure'] < threshold, 'conf_structure'] = 0.0
    
    # 6. Clean up dead rows
    if has_labels:
        mask_to_keep = (df['conf_sequence'] > 0) | (df['conf_structure'] > 0) | (df['conf_protgoat'] > 0) | (df['label'] == 1.0)
    else:
        mask_to_keep = (df['conf_sequence'] > 0) | (df['conf_structure'] > 0) | (df['conf_protgoat'] > 0)
        
    df_trimmed = df[mask_to_keep].copy()
    
    # Calculate "After" stats
    total_struct_after = (df_trimmed['conf_structure'] > 0).sum()
    rows_after = len(df_trimmed)
    
    print("\n--- Thresholding Results ---")
    print(f"  Structure predictions BEFORE: {total_struct_before:,}")
    print(f"  Structure predictions AFTER:  {total_struct_after:,} (Threshold: {threshold})")
    print(f"  Total file rows dropped:      {rows_before - rows_after:,} rows removed")
    
    # 7. Save the new Trimmed Merged File (No header)
    print(f"\nSaving trimmed merged file to: {os.path.basename(output_merged)}")
    df_trimmed.to_csv(output_merged, sep='\t', index=False, header=False)
    
    # 8. Isolate and save Structure-only predictions (3 columns, no header)
    print(f"Saving structure-only file to: {os.path.basename(output_struct)}")
    struct_only = df_trimmed[df_trimmed['conf_structure'] > 0][['protein_id', 'go_term', 'conf_structure']]
    struct_only.to_csv(output_struct, sep='\t', index=False, header=False)
    print("\n")

# --- Define Paths ---
base_dir = '/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/final_data/'

test_in = base_dir + 'test_merged.tsv'
train_in = base_dir + 'train_merged.tsv'

# New Output Paths
test_out_merged = base_dir + 'test_merged_trimmed.tsv'
train_out_merged = base_dir + 'train_merged_trimmed.tsv'

test_out_struct = base_dir + 'test_structure_only_trimmed.tsv'
train_out_struct = base_dir + 'train_structure_only_trimmed.tsv'

# --- Run the Script ---
CONFIDENCE_THRESHOLD = 0.05 

trim_structure_and_analyze(test_in, test_out_merged, test_out_struct, threshold=CONFIDENCE_THRESHOLD)
trim_structure_and_analyze(train_in, train_out_merged, train_out_struct, threshold=CONFIDENCE_THRESHOLD)

Processing: test_merged.tsv


In [ ]:
import pandas as pd
import os

def analyze_trimmed_predictions_per_method(tsv_path):
    print(f"\n{'='*50}")
    print(f"FILE: {os.path.basename(tsv_path)}")
    print(f"{'='*50}")
    
    # 1. Load Data
    compression = 'gzip' if tsv_path.endswith('.gz') else None
    df = pd.read_csv(tsv_path, compression=compression, sep='\t', header=None, low_memory=False)
    
    # 2. Handle 5 vs 6 columns dynamically
    if df.shape[1] == 6:
        df.columns = ['protein_id', 'go_term', 'conf_sequence', 'conf_structure', 'conf_protgoat', 'label']
        conf_cols = ['conf_sequence', 'conf_structure', 'conf_protgoat']
    elif df.shape[1] == 5:
        df.columns = ['protein_id', 'go_term', 'conf_sequence', 'conf_structure', 'conf_protgoat']
        conf_cols = ['conf_sequence', 'conf_structure', 'conf_protgoat']
    else:
        print(f"Unexpected number of columns ({df.shape[1]}). Skipping.")
        return

    # 3. Force columns to numeric to prevent the string/int TypeError
    for col in conf_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    # 4. Calculate stats per method
    for col in conf_cols:
        # A prediction counts ONLY if it is greater than 0
        has_prediction = (df[col] > 0.0)
        
        # Group by protein_id and sum the True/False values (True counts as 1)
        predictions_per_protein = has_prediction.groupby(df['protein_id']).sum()
        
        print(f"\n--- Method: {col.upper()} ---")
        print(f"  Total Predictions Overall: {has_prediction.sum():,}")
        print(f"  Min per protein:    {predictions_per_protein.min()}")
        print(f"  Max per protein:    {predictions_per_protein.max()}")
        print(f"  Mean per protein:   {round(predictions_per_protein.mean(), 2)}")
        print(f"  Median per protein: {predictions_per_protein.median()}")

# --- Define Paths to the TRIMMED files ---
base_dir = '/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/final_data/'

test_trimmed = base_dir + 'test_merged_trimmed.tsv'
train_trimmed = base_dir + 'train_merged_trimmed.tsv'

# --- Run the Analysis ---
print("Analyzing trimmed datasets...")
analyze_trimmed_predictions_per_method(test_trimmed)
analyze_trimmed_predictions_per_method(train_trimmed)

In [ ]:
import pandas as pd

# Your exact file path
labels_path = '/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/data/final_data/train_terms.tsv'

# Load the file. Using header=None just in case it doesn't have column names, 
# but if it does (like 'EntryID' and 'term'), it will just count the header as a row, 
# which won't meaningfully change a dataset of this size.
try:
    # First, try to read assuming there is a header
    df_labels = pd.read_csv(labels_path, sep='\t')
    
    # If the columns aren't standard, we'll just grab them by index position
    protein_col = df_labels.columns[0]
    
    # Group by the first column (protein) and count occurrences
    terms_per_protein = df_labels.groupby(protein_col).size()
    
    print("--- Ground Truth Label Statistics ---")
    print(f"  Total distinct proteins:   {len(terms_per_protein):,}")
    print(f"  Total GO term annotations: {len(df_labels):,}")
    print(f"  Average terms per protein: {round(terms_per_protein.mean(), 2)}")
    print(f"  Median terms per protein:  {terms_per_protein.median()}")
    print(f"  Max terms on one protein:  {terms_per_protein.max()}")
    print(f"  Min terms on one protein:  {terms_per_protein.min()}")

except Exception as e:
    print(f"Error loading file: {e}")

--- Ground Truth Label Statistics ---
  Total distinct proteins:   142,246
  Total GO term annotations: 5,363,863
  Average terms per protein: 37.71
  Median terms per protein:  24.0
  Max terms on one protein:  815
  Min terms on one protein:  2
